# Skill Extraction with Qwen2.5-3B-Instruct

Extracts technical skills from parent vacancies using an open-source LLM.  
Output: a checkpoint JSON mapping `global_id` → list of raw skill strings.  
Normalization and filtering are deferred to a separate merge step.

**Runtime:** Kaggle, GPU T4, ~10 hours for the full dataset.  
**Resumable:** checkpoints are written to storage every 200 records, so the run can survive disconnects.

## 1. Setup

Mount Google Drive and define I/O paths.

In [ ]:
import os, re, json, time
import pandas as pd

INPUT_PATH      = 'path/to/step2_master_vacancies_clustered.parquet'
CHECKPOINT_PATH = 'path/to/step3_skills_checkpoint.json'

## 2. Load Data

Load the deduplicated parquet, keep only parent vacancies, and prepare the input text.

In [ ]:
!pip install -q pyarrow
df = pd.read_parquet(INPUT_PATH)
df_p = df[df['is_parent'] == True].reset_index(drop=True).copy()
df_p['description'] = df_p['description'].fillna('').astype(str)
df_p['title']       = df_p['title'].fillna('').astype(str)
df_p['text']        = (df_p['title'] + '. ' + df_p['description']).str.slice(0, 3000)

print(f'Parents to process: {len(df_p):,}')

## 3. Load the Model

Qwen2.5-3B-Instruct is open-weight, multilingual, and fits comfortably on a single T4 in fp16 (~6 GB VRAM).

> **Important:** before running this cell, enable the GPU via `Runtime → Change runtime type → T4 GPU`.

In [ ]:
!pip install -q transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), 'GPU not active. Runtime → Change runtime type → T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)
model.eval()

## 4. Extraction Function

The system prompt asks for canonical skill names directly, which keeps post-processing minimal.  
Output is parsed as a JSON array; malformed responses yield an empty list rather than failing the batch.

In [ ]:
SYSTEM_PROMPT = (
    'You are a precise skill-extraction tool. Read a job description and output ONLY a valid '
    'JSON array of technical skills mentioned in it. '
    'Include: programming languages, frameworks, libraries, databases, cloud platforms, '
    'DevOps tools, BI/analytics tools, testing tools, methodologies (Agile, Scrum, TDD). '
    'Exclude: human languages, soft skills, job titles, generic words. '
    'Use canonical names: "JavaScript" (not "JS"), "PostgreSQL" (not "Postgres"), '
    '"Kubernetes" (not "k8s"), ".NET" (not "dotnet"). '
    'Output a single JSON array, nothing else.'
)

def build_prompt(text: str) -> str:
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Job description:\n{text}\n\nSkills JSON:'}
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_json_array(text: str) -> list:
    m = re.search(r'\[[^\[\]]*?\]', text, flags=re.DOTALL)
    if not m:
        return []
    try:
        arr = json.loads(m.group(0))
        if isinstance(arr, list):
            return [str(s).strip() for s in arr if isinstance(s, (str, int, float)) and str(s).strip()]
    except json.JSONDecodeError:
        pass
    return []

def extract_skills_batch(texts: list, max_new_tokens: int = 200) -> list:
    prompts = [build_prompt(t) for t in texts]
    inputs = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )
    results = []
    for j in range(len(texts)):
        gen_tokens = out[j, inputs['input_ids'].shape[1]:]
        decoded = tok.decode(gen_tokens, skip_special_tokens=True)
        results.append(parse_json_array(decoded))
    return results

## 5. Resume from Checkpoint

If a previous run was interrupted, load the existing checkpoint and skip already-processed records.

In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        done = json.load(f)
    print(f'Resuming: {len(done):,} records already processed')
else:
    done = {}
    print('Starting from scratch')

## 6. Main Extraction Loop

Processes vacancies in batches of 8. The OOM fallback retries any failing batch one record at a time, which protects the run against unusually long descriptions.  
Progress is written to Google Drive every 200 records — if the Colab session disconnects, simply rerun cells 1, 3, 5, 6 and it continues from the last checkpoint.

In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 8
SAVE_EVERY = 200

to_process = df_p[~df_p['global_id'].isin(done)].reset_index(drop=True)
print(f'Remaining: {len(to_process):,}')

session_start = time.time()
session_processed = 0
since_save = 0

for i in tqdm(range(0, len(to_process), BATCH_SIZE)):
    batch = to_process.iloc[i:i + BATCH_SIZE]
    try:
        skills = extract_skills_batch(batch['text'].tolist())
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        skills = [extract_skills_batch([t])[0] for t in batch['text'].tolist()]

    for gid, sk in zip(batch['global_id'], skills):
        done[gid] = sk
    since_save        += len(batch)
    session_processed += len(batch)

    if since_save >= SAVE_EVERY:
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump(done, f)
        elapsed = time.time() - session_start
        rate    = session_processed / elapsed if elapsed > 0 else 0
        remaining = len(to_process) - (i + BATCH_SIZE)
        eta_h   = (remaining / rate) / 3600 if rate > 0 else 0
        tqdm.write(f'Saved {len(done):,} | {rate:.2f} rec/s | ETA {eta_h:.2f}h')
        since_save = 0

with open(CHECKPOINT_PATH, 'w') as f:
    json.dump(done, f)
print(f'Done. {len(done):,} records saved to {CHECKPOINT_PATH}')

## 7. Quick Preview

Inspect the raw output before normalization. Expect some inconsistency (`Postgres` vs `PostgreSQL`, version suffixes, occasional noise) — these will be cleaned in the merge step.

In [ ]:
from collections import Counter

ctr = Counter()
for skills in done.values():
    ctr.update(skills)

print(f'Records with extracted skills:  {len(done):,}')
print(f'Total skill mentions:           {sum(ctr.values()):,}')
print(f'Unique raw skill strings:       {len(ctr):,}')

print('\nTop 50 raw skills (pre-normalization):')
for sk, cnt in ctr.most_common(50):
    print(f'  {sk:<32} {cnt:,}')